In [1]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad

In [2]:
# experimental data
save_folder = 'run7'
n_points = 10000

lower_factor = 0.99
upper_factor = 2 - lower_factor

# Load experimental data
atlas_data = pd.read_csv('../../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

/tmp/ipykernel_7608/4196082941.py:9: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  atlas_data = pd.read_csv('../../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
/tmp/ipykernel_7608/4196082941.py:10: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  totem_data = pd.read_csv('../../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)


In [3]:
# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'epsilon': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    },
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491,
            'a2': 2.77
        },
        'pl':{
            'epsilon': 0.0892,
            'mg': 0.447,
            'a1': 1.689,
            'a2': 1.7
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'   

def get_parameters_with_variations(ensemble_parameters, ensemble_name, model_type, lower_factor=lower_factor, upper_factor=upper_factor):
    # Obtém os parâmetros iniciais
    initial_params = ensemble_parameters[ensemble_name][model_type]
    
    # Cria as variações
    initial_params_low = {k: v * lower_factor for k, v in initial_params.items()}
    initial_params_high = {k: v * upper_factor for k, v in initial_params.items()}
    
    return initial_params, initial_params_low, initial_params_high

# Get parameters for selected configuration
initial_params_pl_atlas = ensemble_parameters[ensemble_atlas][pl_model_type]

# Para Atlas
initial_params_pl_atlas, initial_params_low_pl_atlas, initial_params_high_pl_atlas = \
    get_parameters_with_variations(ensemble_parameters, ensemble_atlas, pl_model_type)




In [4]:
# def model functions 
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q
    qk_cos = np.sqrt(np.abs(q)) * np.abs(k) * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(np.abs(q)) * np.abs(k) * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  

def differential_sigma(amp_value, s):
    amp_squared = amp_value * amp_value.conjugate()
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323


In [5]:
def full_int(mg, a1, a2, m2_func, q_val, sqrt_s):
    q_val = np.atleast_1d(q_val)
    results = []

    for q in q_val:                      # <- q is the correct variable
        def integrand(y, x, mg, a1, a2, m2_func, q):
            k = sqrt_s * x
            phi = 2 * np.pi * y
            jacobian = 2 * np.pi * sqrt_s
            return k * (
                T_1(k, q, phi, mg, a1, a2, m2_func)   # <-- FIXED: q instead of q_val
                - T_2(k, q, phi, mg, a1, a2, m2_func)
            ) * jacobian

        def inner_integral(x):
            integral_real = fixed_quad(
                lambda y: np.real(integrand(y, x, mg, a1, a2, m2_func, q)),
                0, 1, n=n_points
            )[0]
            integral_imag = fixed_quad(
                lambda y: np.imag(integrand(y, x, mg, a1, a2, m2_func, q)),
                0, 1, n=n_points
            )[0]
            return integral_real + 1j * integral_imag

        integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
        results.append(integral_value)

    return np.array(results) if len(results) > 1 else results[0]


In [6]:
# def model function
def model_function(x, eps, mg, a1, a2, sqrt_s, model_type='log'):

    # Definindo os parâmetros específicos do modelo
    params = {
        'epsilon': eps,
        'mg': mg,
        'a1': a1,
        'a2': a2
    }
    
    # Escolhendo a massa conforme o modelo
    m2 = m2_log if model_type == 'log' else m2_pl
    
    dif_sigma_lst = []
    
    for q2 in x:
        t = -q2
        
        integral_value = full_int(mg, a1, a2, m2, q2, sqrt_s)

        diff_T = integral_value
        s = sqrt_s ** 2
        amp_value = amp_calculation(diff_T, s, params['epsilon'], t)
        dif_sigma_value = differential_sigma(amp_value, s)
        dif_sigma_lst.append(dif_sigma_value)
    
    return np.array(dif_sigma_lst)

In [7]:
# set cost and minimize
def model_7(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=7000, model_type='pl')

def model_8(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=8000, model_type='pl')

def model_13(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=13000, model_type='pl')


chi2_7  = LeastSquares(x_7_atlas,  y_7_atlas,  yerr_7_atlas,  model_7)
chi2_8  = LeastSquares(x_8_atlas,  y_8_atlas,  yerr_8_atlas,  model_8)
chi2_13 = LeastSquares(x_13_atlas, y_13_atlas, yerr_13_atlas, model_13)


chi2_total = chi2_7 + chi2_8 + chi2_13


minuit_born = Minuit(
    chi2_total,
    mg = 0.421,
    a1 = 1.517,
    a2 = 2.05,
    eps = 0.0753
)

minuit_born.migrad()
minuit_born.hesse()


/home/victor/miniconda3/envs/myroot/lib/python3.9/site-packages/iminuit/minuit.py:2781: ComplexWarning: Casting complex values to real discards the imaginary part
  fm = migrad(ncall, tolerance)
/home/victor/miniconda3/envs/myroot/lib/python3.9/site-packages/iminuit/minuit.py:1467: ComplexWarning: Casting complex values to real discards the imaginary part
  hesse(self._fcn, fm, replace_none(ncall, 0), self._fmin.edm_goal)


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 25.93 (χ²/ndof = 0.2)      │              Nfcn = 341              │
│ EDM = 3.1e-06 (Goal: 0.0002)     │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ eps  │  0.0616   │  0.0022   │            │            │         │         │       │
│ 1 │ mg   │   0.389   │   0.005   │            │            │         │         │       │
│ 2 │ a1   │   1.49    │   0.05    │            │            │         │         │       │
│ 3 │ a2   │   2.16    │   0.31    │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬─────────────────────────────────────┐
│     │      eps       mg       a1       a2 │
├─────┼─────────────────────────────────────┤
│ eps │ 4.86e-06    10e-6    54e-6  -157e-6 │
│  mg │    10e-6 2.41e-05 0.056e-3 0.100e-3 │
│  a1 │    54e-6 0.056e-3   0.0023  -0.0136 │
│  a2 │  -157e-6 0.100e-3  -0.0136   0.0956 │
└─────┴─────────────────────────────────────┘

In [8]:
# Calculates and plot dif sigma 
diff_t_born = []
def get_dif_sigma(epsilon, mg, a1, a2, mg_model):

    sqrt_s = 7000
    scale = 1  # caso único
    start_q2 = 0.006
    max_q2   = 0.204
    q2_step  = 0.001

    lst_q2 = []
    lst_dif_sigma = []

    q2 = start_q2
    while q2 <= max_q2:
        t = -q2

        integral_value = full_int(mg, a1, a2, mg_model, q2, sqrt_s)


        diff_T = integral_value
        # print(diff_T)
        diff_t_born.append(diff_T)
        # print(f"q2: {q2}, diff_T: {diff_T}")

        s          = sqrt_s**2
        amp_value  = amp_calculation(diff_T, s, epsilon, t)
        # print(amp_value) 
        dif_sigma  = differential_sigma(amp_value, s) * scale
        # print(dif_sigma)

        lst_q2.append(q2)
        lst_dif_sigma.append(dif_sigma)

        q2 += q2_step
        # print(q2)

    return {sqrt_s: (lst_q2, lst_dif_sigma)}

#for pl atlas
dif_sigma_pl_atlas = get_dif_sigma(
    minuit_born.values['eps'],
    minuit_born.values['mg'], 
    minuit_born.values['a1'],
    minuit_born.values['a2'],
    m2_pl
)

dif_sigma_pl_atlas_7_q2 = dif_sigma_pl_atlas[7000][0]
dif_sigma_pl_atlas_7_values = dif_sigma_pl_atlas[7000][1]


def add_differential_trace(fig, x, y, label, color='red', mg_model= 'log', legend=True, size = 4, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=size),
        name=f'{label}, {mg_model}',
        showlegend=legend,

    ))

def add_data_trace(fig, x, y, y_error, scale=1.0, color='black', size=4,
                           name=None, show_label=True, mode='markers'):
    fig.add_trace(go.Scatter(
        x=x,
        y=y * scale,
        mode=mode,
        marker=dict(color=color, size=size),
        error_y=dict(
            type='data',
            array=y_error * scale,
            visible=True
        ),
        name=name if show_label else None,
        showlegend=show_label
    ))

fig_atlas = go.Figure()


# for pl atlas
add_differential_trace(fig_atlas, dif_sigma_pl_atlas_7_q2, np.real(dif_sigma_pl_atlas_7_values),label='7 TeV', color='blue', mg_model='pl')

#data points
add_data_trace(fig_atlas, x_7_atlas, y_7_atlas, yerr_7_atlas, name='ATLAS 7 TeV', show_label=True, mode='markers')

# Atualiza layout
fig_atlas.update_layout(
    title='dσ/dt vs. |t| - Log and PL models in ATLAS',
    xaxis_title='|t| (GeV²)',
    yaxis_title='dσ/dt (mb/GeV²)',
    yaxis_type='log',
    legend_title='Mass Model',
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_atlas.update_xaxes(gridcolor='lightgray')
fig_atlas.update_yaxes(gridcolor='lightgray')

# fig_atlas.show(renderer='browser')


In [9]:
# # PLOT BORN SIGMA TOT =============================================================
# # 
# # =============================================================


data_sigma_tot_atlas = pd.read_csv(
    "../../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70
)

x_sigma_tot_atlas = data_sigma_tot_atlas[0].to_numpy()
y_sigma_tot_atlas = data_sigma_tot_atlas[1].to_numpy()
y_error_sigma_tot_atlas = data_sigma_tot_atlas[2].to_numpy()

lst_born_amp = []

start_sqrt_s = 1
max_sqrt_s = 13010
step = 100

def add_total_trace(fig, x, y, color='red', label='', line_style='solid', legend=True, size = 3, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width, dash=line_style),
        marker=dict(size=size),
        name = label, 
        showlegend=legend
    ))

def get_sigma_tot(epsilon, mg, a1, a2, mg_model):

    lst_sigma_tot = []
    lst_sqrt_s = []

    sqrt_s = start_sqrt_s

    while sqrt_s <= max_sqrt_s:

        s = sqrt_s ** 2

        integral_value = full_int(mg, a1, a2, mg_model, 0.0, sqrt_s) 

        born_amp = amp_calculation(integral_value, s, epsilon, 0)
        # print(born_amp)
        lst_born_amp.append(born_amp)
        
        lst_sigma_tot.append(sigma_tot(
            amp_calculation(integral_value, s, epsilon, 0), s))
        
        lst_sqrt_s.append(sqrt_s)
        sqrt_s += step
    return lst_sigma_tot, lst_sqrt_s

##-----------------------------------------------------------------------------------------------

sigma_tot_pl_atlas = get_sigma_tot(
    minuit_born.values['eps'],
    minuit_born.values['mg'],
    minuit_born.values['a1'],
    minuit_born.values['a2'],
    m2_pl
)

sigma_tot_pl_atlas_values = sigma_tot_pl_atlas[0]
lst_sqrt_s = sigma_tot_pl_atlas[1]



fig = go.Figure()

add_total_trace(fig, lst_sqrt_s, sigma_tot_pl_atlas_values, color='blue', label='PL Atlas', line_style='solid')

#-----------------------------------------------------------------------------------------------
#----

add_data_trace(fig, x_sigma_tot_atlas, y_sigma_tot_atlas, y_error_sigma_tot_atlas, name='ATLAS', show_label=True, mode='markers')

fig.update_layout(
    title = 'σ_tot vs. √s - Ensemble Atlas and Totem in Log and PL model',
    xaxis=dict(
        title='√s [GeV]',
        type='log',
        range=[np.log10(2000), np.log10(14000)],
    ),
    yaxis=dict(
        title='σ_tot [mb]',
        range=[80, 125]
    ),
    showlegend=True,
    legend=dict(
        title='Ensembles'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)
    
fig.update_xaxes(gridcolor='lightgray')
fig.update_yaxes(gridcolor='lightgray')

# fig.show(renderer="browser")

/tmp/ipykernel_7608/24644038.py:6: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead



In [10]:
# from scipy.integrate import quad  # kept for compatibility if used elsewhere

# lst_chi = []
# lst_amp_eik = []
# lst_diff_sigma = []

# lst_q_integration = np.linspace(0, 5, 1000)
# lst_b_integration = np.linspace(0, 15, 1000)

# # step size for Riemann sum
# dq = lst_q_integration[1] - lst_q_integration[0]

# # midpoints for improved Riemann sum accuracy
# q_midpoints = lst_q_integration[:-1] + dq / 2

# for b_val in lst_b_integration:

#     chi_sum = 0

#     for q_val in q_midpoints:

#         q2_val = q_val**2
#         sqrt_s = 7000

#         s = sqrt_s ** 2
#         t = -q2_val

#         diff_t = full_int(
#             minuit_born.values['mg'],
#             minuit_born.values['a1'],
#             minuit_born.values['a2'],
#             m2_pl,
#             q2_val,
#             sqrt_s
#         )

#         born_amp = amp_calculation(diff_t, s, minuit_born.values['eps'], t)

#         chi_val = (1/s) * q_val * j0(b_val * q_val) * born_amp

#         # Midpoint Riemann sum contribution
#         chi_sum += chi_val * dq

#     print(chi_sum)

#     lst_chi.append(chi_sum)


In [11]:
# import numpy as np
# from scipy.integrate import quad
# from scipy.special import j0

# lst_chi = []
# lst_amp_eik = []

# # Integration limits
# q_min, q_max = 0, 0.4
# b_min, b_max = 0, 20

# # Integration grid for outer b loop
# n_points_b = 200
# lst_b = np.linspace(b_min, b_max, n_points_b)

# sqrt_s = 13000
# s = sqrt_s ** 2


# # Define integrand in q for given b
# def chi_integrand_q(q_val, b_val):
#     q2_val = q_val ** 2
#     t = -q2_val

#     diff_t = full_int(
#         minuit_born.values['mg'],
#         minuit_born.values['a1'],
#         minuit_born.values['a2'],
#         m2_pl,
#         q2_val,
#         sqrt_s
#     )

#     born_amp = amp_calculation(diff_t, s, minuit_born.values['eps'], t)
#     return (1 / s) * q_val * j0(b_val * q_val) * born_amp  # complex-valued


# # Outer integration in b (explicit loop)
# def chi_integrand_b(b_val):
#     # Integrate real and imaginary parts separately
#     real_part = lambda q: np.real(chi_integrand_q(q, b_val))
#     imag_part = lambda q: np.imag(chi_integrand_q(q, b_val))

#     chi_real, _ = quad(real_part, q_min, q_max)
#     chi_imag, _ = quad(imag_part, q_min, q_max)

#     return chi_real + 1j * chi_imag


# # Perform full integration in b
# for b_val in lst_b:
#     chi_sum = chi_integrand_b(b_val)
#     print(chi_sum)
#     lst_chi.append(chi_sum)


In [12]:
# from scipy.integrate import quad
# import numpy as np
# from scipy.special import j0

# b_max = 30
# q_min, q_max = 0, 10

# lst_chi = []
# lst_b_integration = np.linspace(0, b_max, 100)

# sqrt_s = 7000
# s = sqrt_s ** 2


# # Loop over b points (so we can store chi(b) values)
# for b_val in lst_b_integration:

#     def integrand_q(q_val):
#         q2_val = q_val ** 2
#         t = -q2_val

#         diff_t = full_int(
#             minuit_born.values['mg'],
#             minuit_born.values['a1'],
#             minuit_born.values['a2'],
#             m2_pl,
#             q2_val,
#             sqrt_s
#         )
#         print(f'diff_t: {diff_t} q2_val: {q2_val}')
#         print(50*'-')

#         born_amp = amp_calculation(diff_t, s, minuit_born.values['eps'], t)

#         # Keep complex amplitude fully represented
#         chi_val = (1/s) * q_val * j0(b_val * q_val) * born_amp
#         return chi_val

#     # Separate integration for real and imaginary parts for numerical stability
#     # chi_real, err_real = quad(lambda q: np.real(integrand_q(q)), q_min, q_max, limit=300, epsabs=1e-8, epsrel=1e-6)
#     chi_imag, err_imag = quad(lambda q: np.imag(integrand_q(q)), q_min, q_max)

#     chi_val = 0 + 1j * chi_imag
#     lst_chi.append(chi_val)

#     # print(f"b = {b_val:.3f} -> chi_val = {chi_val}, err_imag = {err_imag}")


In [13]:
# # from scipy.integrate import quad  # kept for compatibility if used elsewhere

# # Define lists
# lst_chi = []
# lst_b = []
# lst_amp_eik = []
# lst_diff_sigma = []

# # Integration ranges
# q_min, q_max = 0, 30
# b_min, b_max = 19, 20

# # Step sizes
# n_q = 1000
# n_b = 200
# dq = (q_max - q_min) / n_q
# db = (b_max - b_min) / n_b

# # Midpoints for q integration
# q_mid = q_min + dq / 2

# # Constants
# sqrt_s = 7000
# s = sqrt_s ** 2

# # Initialize b
# b_val = b_min

# # Loop over b using while
# while b_val <= b_max:
#     chi_sum = 0
#     q_val = q_min

#     # Inner loop over q using while
#     while q_val <= q_max - dq:
#         q_midpoint = q_val + dq / 2
#         q2_val = q_midpoint ** 2
#         t = -q2_val

#         diff_t = full_int(
#             minuit_born.values['mg'],
#             minuit_born.values['a1'],
#             minuit_born.values['a2'],
#             m2_pl,
#             q2_val,
#             sqrt_s
#         )

#         born_amp = amp_calculation(
#             diff_t,
#             s,
#             minuit_born.values['eps'],
#             t
#         )

#         chi_val = (1 / s) * q_midpoint * j0(b_val * q_midpoint) * born_amp
#         chi_sum += chi_val * dq

#         q_val += dq  # increment q

#     print(chi_sum)
#     lst_chi.append(chi_sum)

#     b_val += db  # increment b

#     # Append values to lists
#     lst_b.append(b_val)

In [14]:
# import numpy as np
# import matplotlib.pyplot as plt
# from scipy.integrate import fixed_quad
# from scipy.special import j0  # Bessel function J_0


# # Model functions
# def m2_log(q2, mg):
#     lambda_squared = Lambda ** 2
#     rho_mg_squared = rho * mg ** 2
#     ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
#     return mg ** 2 * ratio ** (-1 - gamma_1)

# def m2_pl(q2, mg):
#     lambda_squared = Lambda ** 2
#     rho_mg_squared = rho * mg ** 2
#     ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
#     return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

# def G_p(q2, a1, a2):
#     return np.exp(-(a1 * q2 + a2 * q2 ** 2))

# def alpha_D(q2, mg, m2_func):
#     m2 = m2_func(q2, mg)
#     return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

# def T_1(k, q, phi, mg, a1, a2, m2_func):
#     q2 = q
#     qk_cos = np.sqrt(np.abs(q)) * np.abs(k) * np.cos(phi)
#     qk_plus_squared = q2 / 4 + qk_cos + k ** 2
#     qk_minus_squared = q2 / 4 - qk_cos + k ** 2
#     alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
#     alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
#     G0 = G_p(q2, a1, a2)
#     return alpha_D_plus * alpha_D_minus * G0 ** 2

# def T_2(k, q, phi, mg, a1, a2, m2_func):
#     q2 = q 
#     qk_cos = np.sqrt(np.abs(q)) * np.abs(k) * np.cos(phi)
#     qk_plus_squared = q2 / 4 + qk_cos + k ** 2
#     qk_minus_squared = q2 / 4 - qk_cos + k ** 2
#     alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
#     alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
#     factor = q2 + 9 * abs(k ** 2 - q2 / 4)
#     G0 = G_p(q2, a1, a2)
#     G_minus = G_p(factor, a1, a2)
#     return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

# def amp_calculation(diff_T, s, epsilon, t):
#     alpha_pomeron = 1.0 + epsilon + alpha_prime * t
#     regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
#     return 1j * 8 * regge_factor * diff_T

# def full_int(mg, a1, a2, m2_func, q_val, sqrt_s):
#     q_val = np.atleast_1d(q_val)
#     results = []
#     for q in q_val:
#         def integrand(y, x, mg, a1, a2, m2_func, q):
#             k = sqrt_s * x
#             phi = 2 * np.pi * y
#             jacobian = 2 * np.pi * sqrt_s
#             return k * (
#                 T_1(k, q, phi, mg, a1, a2, m2_func)
#                 - T_2(k, q, phi, mg, a1, a2, m2_func)
#             ) * jacobian
        
#         def inner_integral(x):
#             integral_real = fixed_quad(
#                 lambda y: np.real(integrand(y, x, mg, a1, a2, m2_func, q)),
#                 0, 1, n=n_points
#             )[0]
#             integral_imag = fixed_quad(
#                 lambda y: np.imag(integrand(y, x, mg, a1, a2, m2_func, q)),
#                 0, 1, n=n_points
#             )[0]
#             return integral_real + 1j * integral_imag
        
#         integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
#         results.append(integral_value)
#     return np.array(results) if len(results) > 1 else results[0]

# # Eikonal function calculation using equation 23
# def calculate_chi(b_val, sqrt_s, mg, a1, a2, m2_func, q_max=30, n_q_points=50):
#     """
#     Calculate χ(s,b) using Fourier-Bessel transform (Eq. 23)
#     χ(s,b) = (1/s) ∫₀^∞ q dq J₀(bq) A_Born(s,t)
#     where t = -q²
#     """
#     s = sqrt_s ** 2
#     q_values = np.linspace(0.01, q_max, n_q_points)  # avoid q=0 for numerical stability
    
#     # Calculate A_Born for each q value
#     diff_T_values = full_int(mg, a1, a2, m2_func, q_values**2, sqrt_s)
#     A_Born_values = amp_calculation(diff_T_values, s, epsilon, -q_values**2)
    
#     # Integrand: q * J_0(b*q) * A_Born(s, t=-q²)
#     integrand_values = q_values * j0(b_val * q_values) * A_Born_values
    
#     # Numerical integration using trapezoidal rule
#     chi = np.trapz(integrand_values, q_values) / s
    
#     return chi

# # Main calculation
# if __name__ == "__main__":
#     # Parameters
#     sqrt_s = 7000.0  # GeV
#     mg = minuit_born.values['mg']  # GeV (example value)
#     a1 = minuit_born.values['a1']  # GeV^-2 (example value)
#     a2 = minuit_born.values['a2']  # GeV^-4 (example value)
#     epsilon = minuit_born.values["eps"]
    
#     # Impact parameter range
#     b_values = np.linspace(0.01, 20, 50)  # GeV^-1, avoid b=0 for numerical stability
    
#     print(f"Calculating χ(b) for √s = {sqrt_s} GeV")
#     print(f"Parameters: mg={mg}, a1={a1}, a2={a2}")
#     print("This may take a few minutes...\n")
    
#     # Calculate chi for each b value
#     chi_values = []
#     for i, b in enumerate(b_values):
#         chi = calculate_chi(b, sqrt_s, mg, a1, a2, m2_pl, q_max=10, n_q_points=500)
#         chi_values.append(chi)
#         if (i+1) % 5 == 0:
#             print(f"Progress: {i+1}/{len(b_values)} b values calculated")
    
#     chi_values = np.array(chi_values)
    
#     # Plot results
#     plt.figure(figsize=(12, 5))
    
#     # Plot imaginary part
#     plt.subplot(1, 2, 2)
#     plt.plot(b_values, np.imag(chi_values), 'r-', linewidth=2)
#     plt.xlabel('Impact parameter b [GeV⁻¹]', fontsize=12)
#     plt.ylabel('Im χ(s,b)', fontsize=12)
#     plt.title(f'Imaginary part of Eikonal Function\n√s = {sqrt_s} GeV', fontsize=13)
#     plt.grid(True, alpha=0.3)
    
#     plt.tight_layout()
#     plt.show()

In [26]:
import numpy as np
from scipy.special import j0
from scipy.integrate import quad

lst_chi = []
lst_amp_eik = []
lst_diff_sigma = []

lst_b_integration = np.linspace(0, 10, 50)

sqrt_s = 7000
s = sqrt_s ** 2

eps_rel = 1e-6
eps_abs = 1e-16

q_max = 1

def chi_integrand(q_val, b_val):
    q2_val = q_val ** 2
    t = -q2_val

    diff_t = full_int(
        minuit_born.values['mg'],
        minuit_born.values['a1'],
        minuit_born.values['a2'],
        m2_pl,
        q2_val,
        sqrt_s
    )

    born_amp = amp_calculation(
        diff_t,
        s,
        minuit_born.values['eps'],
        t
    )

    return (1/s) * q_val * j0(b_val * q_val) * born_amp


for b_val in lst_b_integration:

    # quad doesn't handle complex integrands natively, so split real and imag
    real_part, _ = quad(lambda q: np.real(chi_integrand(q, b_val)), 0, q_max, epsrel=eps_rel, epsabs=eps_abs)
    imag_part, _ = quad(lambda q: np.imag(chi_integrand(q, b_val)), 0, q_max, epsrel=eps_rel, epsabs=eps_abs)

    chi_sum = real_part + 1j * imag_part

    print(f"b = {b_val:.2f}, χ(b) = {np.imag(chi_sum):.4f}")

    lst_chi.append(chi_sum)

b = 0.00, χ(b) = 12.5110
b = 0.20, χ(b) = 12.4980
b = 0.41, χ(b) = 12.4590
b = 0.61, χ(b) = 12.3942
b = 0.82, χ(b) = 12.3042
b = 1.02, χ(b) = 12.1893
b = 1.22, χ(b) = 12.0503
b = 1.43, χ(b) = 11.8880
b = 1.63, χ(b) = 11.7034
b = 1.84, χ(b) = 11.4975
b = 2.04, χ(b) = 11.2715
b = 2.24, χ(b) = 11.0268
b = 2.45, χ(b) = 10.7646
b = 2.65, χ(b) = 10.4866
b = 2.86, χ(b) = 10.1940
b = 3.06, χ(b) = 9.8887
b = 3.27, χ(b) = 9.5721
b = 3.47, χ(b) = 9.2458
b = 3.67, χ(b) = 8.9116
b = 3.88, χ(b) = 8.5711
b = 4.08, χ(b) = 8.2259
b = 4.29, χ(b) = 7.8775
b = 4.49, χ(b) = 7.5276
b = 4.69, χ(b) = 7.1776
b = 4.90, χ(b) = 6.8289
b = 5.10, χ(b) = 6.4831
b = 5.31, χ(b) = 6.1413
b = 5.51, χ(b) = 5.8047
b = 5.71, χ(b) = 5.4745
b = 5.92, χ(b) = 5.1518
b = 6.12, χ(b) = 4.8373
b = 6.33, χ(b) = 4.5320
b = 6.53, χ(b) = 4.2366
b = 6.73, χ(b) = 3.9516
b = 6.94, χ(b) = 3.6777
b = 7.14, χ(b) = 3.4152
b = 7.35, χ(b) = 3.1644
b = 7.55, χ(b) = 2.9256
b = 7.76, χ(b) = 2.6989
b = 7.96, χ(b) = 2.4843
b = 8.16, χ(b) = 2.2818
b

In [27]:
from tkinter import Grid
import plotly.graph_objects as go

chi_imag = np.imag(lst_chi)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=lst_b_integration, y=chi_imag,
        mode='lines+markers',
        name='Im χ(b)',
        marker=dict(symbol='square', size=7, color='tomato'),
        line=dict(color='tomato', width=2)
    )
)

fig.update_layout(
    title=dict(text="Eikonal phase χ(b) vs impact parameter b", font=dict(size=16)),
    xaxis_title="b",
    yaxis_title="Im χ(b)",
    template="plotly_white",
    height=500,
    width=1200
)

fig.show()

In [61]:
# import numpy as np
# from scipy.special import j0

# # Define lists
# lst_chi = []
# lst_b = []
# lst_amp_eik = []
# lst_diff_sigma = []

# # Integration ranges
# q_min, q_max = 0, 5
# b_min, b_max = 0, 20

# # Step sizes
# n_q = 5000  
# n_b = 200
# dq = (q_max - q_min) / n_q
# db = (b_max - b_min) / n_b

# # Precompute q midpoints
# q_vals = q_min + (np.arange(n_q) + 0.5) * dq
# q2_vals = q_vals ** 2
# t_vals = -q2_vals

# # Constants
# sqrt_s = 7000
# s = sqrt_s ** 2

# # Precompute Born amplitudes (depends only on q)
# diff_t_vals = np.array([
#     full_int(
#         minuit_born.values['mg'],
#         minuit_born.values['a1'],
#         minuit_born.values['a2'],
#         m2_pl,
#         q2,
#         sqrt_s
#     )
#     for q2 in q2_vals
# ])
# born_amp_vals = np.array([
#     amp_calculation(
#         diff_t_vals[i],
#         s,
#         minuit_born.values['eps'],
#         t_vals[i]
#     )
#     for i in range(len(q_vals))
# ])

# b_vals = b_min + (np.arange(n_b) + 0.5) * db

# for b_val in b_vals:
#     chi_integrand = (1 / s) * q_vals * j0(b_val * q_vals) * born_amp_vals
#     chi_sum = np.sum(chi_integrand) * dq  # Riemann midpoint sum
#     lst_chi.append(chi_sum)
#     lst_b.append(b_val)
#     print(chi_sum)

In [62]:
fig_chi = go.Figure()
add_total_trace(fig_chi, lst_b_integration, np.imag(lst_chi), color='red', line_style='solid')

fig_chi.update_layout(
    title = r'$Im(\chi) \quad \text{vs.} \quad b \quad \text{(fixed sqrt(s) = 7000 GeV)}$',
    xaxis=dict(
        title='b'
    ),
    # yaxis = dict(
    #     type = "log",
    # ),

    showlegend=True,
    legend=dict(
        title=r'$Im(\chi)$',
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)
    
fig_chi.update_xaxes(gridcolor='lightgray')
fig_chi.update_yaxes(gridcolor='lightgray')

fig_chi.show(renderer="browser")
# fig_chi.write_image('../../../../results/eikonal/b_values_plots/chi_b.pdf', width=1200, height=600)
# fig_chi.write_html('chi_sum_while.html')


Opening in existing browser session.


In [69]:
import numpy as np
from scipy.special import j0

# lists for results
lst_chi = []
lst_amp_eik = []
lst_diff_sigma = []

# integration grids
lst_q_integration = np.linspace(0, 30, 1000)
lst_b_integration = np.linspace(0, 20, 1000)

# Riemann-step
dq = lst_q_integration[1] - lst_q_integration[0]
db = lst_b_integration[1] - lst_b_integration[0]

# midpoints
q_midpoints = lst_q_integration[:-1] + dq / 2
b_midpoints = lst_b_integration[:-1] + db / 2

sqrt_s = 7000
s = sqrt_s ** 2


eik_amp_sum = 0

for b_val in lst_b_integration:

    chi_sum = 0

    for q_val in q_midpoints:

        # physical variables
        q2_val = q_val**2
        t = -q2_val                           # t = -q^2

        # ------------------------------------------
        # FIX #1: full_int must receive q, not q^2
        # ------------------------------------------
        diff_t = full_int(
            minuit_born.values['mg'],
            minuit_born.values['a1'],
            minuit_born.values['a2'],
            m2_pl,
            q2_val,            # <-- FIXED (was q2_val)
            sqrt_s
        )

        # Born amplitude
        born_amp = amp_calculation(
            diff_t,
            s,
            minuit_born.values['eps'],
            t
        )

        # χ(b) integrand
        chi_val = (1/s) * q_val * j0(b_val * q_val) * born_amp

        chi_sum += chi_val * dq  # Riemann midpoint sum

    print(chi_sum)
    factor = 1 - np.exp(1j * chi_sum)
    eik_amp_value = (1j * s) * b_val * j0(b_val * np.sqrt(0.006)) * factor
    eik_amp_sum += eik_amp_value * db


    lst_chi.append(chi_sum)



diff_sigma_val = (np.pi / (s**2)) * (eik_amp_sum * eik_amp_sum.conjugate()) * 0.39
print(diff_sigma_val)

12.52029040971873j
12.520165012123307j
12.51978882663363j
12.51916187513884j
12.518284194118612j
12.517155834639698j
12.515776862351277j
12.514147357478812j
12.51226741481662j
12.510137143719088j
12.507756668090533j
12.505126126373709j
12.502245671536992j
12.49911547106015j
12.495735706918866j
12.492106575567862j
12.488228287922677j
12.484101069340145j
12.479725159597495j
12.475100812870172j
12.47022829770828j
12.46510789701172j
12.459739908003979j
12.454124642204654j
12.448262425400637j
12.442153597615903j
12.435798513080162j
12.429197540195993j
12.422351061504923j
12.415259473651938j
12.407923187348912j
12.400342627336656j
12.392518232345706j
12.384450455055804j
12.376139762054146j
12.367586633792328j
12.358791564542047j
12.349755062349542j
12.340477648988752j
12.330959859913303j
12.321202244207162j
12.31120536453409j
12.300969797085942j
12.290496131529588j
12.279784970952786j
12.26883693180873j
12.257652643859391j
12.24623275011783j
12.234577906789061j
12.222688783209952j
12.2105660